  TASK 1 : "TEXT GENERATION WITH GPT-2"


In [6]:
with open("custom_dataset.txt", "w") as f:
    f.write("""The future of artificial intelligence lies in seamless human-machine collaboration. Robots and algorithms are becoming an integral part of everyday life, transforming industries from healthcare to transportation. As technology advances, ethical considerations about AI decision-making continue to grow in importance. In a world driven by data, machine learning models are the engines of innovation. From virtual assistants to autonomous vehicles, AI is redefining how we interact with the world. The rise of quantum computing promises to accelerate AI capabilities beyond current imagination. In digital economies, AI-driven personalization is reshaping consumer behavior and expectations. Technological singularity refers to a point where AI surpasses human intelligence, sparking debates among scientists and futurists. Deep learning models are inspired by the structure and function of the human brain's neural networks. Despite challenges, AI holds the potential to solve some of humanity's biggest problems, from climate change to global health crises. """)


In [9]:
# Install the required libraries
!pip install transformers datasets --quiet

import os
os.environ["WANDB_DISABLED"] = "true"

# Import libraries
from transformers import GPT2LMHeadModel, GPT2Tokenizer, TextDataset, DataCollatorForLanguageModeling, Trainer, TrainingArguments
import torch

# Check if GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Load the GPT-2 tokenizer and model
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name).to(device)

# GPT-2 needs a padding token — we set it
tokenizer.pad_token = tokenizer.eos_token
model.resize_token_embeddings(len(tokenizer))

# Upload your dataset.txt file
from google.colab import files
uploaded = files.upload()

# Save the uploaded file name
dataset_path = list(uploaded.keys())[0]

# Create a dataset
def load_dataset(train_file, tokenizer):
    return TextDataset(
        tokenizer=tokenizer,
        file_path=train_file,
        block_size=128
    )

train_dataset = load_dataset(dataset_path, tokenizer)

# Data collator helps during batching
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False
)

# Set training arguments
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    prediction_loss_only=True,
)

# Set up Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)

# Train the model
trainer.train()

# Save the final model and tokenizer
trainer.save_model("./gpt2-finetuned")
tokenizer.save_pretrained("./gpt2-finetuned")


Using device: cuda


/usr/local/lib/python3.11/dist-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Saving custom_dataset.txt to custom_dataset (3).txt


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


('./gpt2-finetuned/tokenizer_config.json',
 './gpt2-finetuned/special_tokens_map.json',
 './gpt2-finetuned/vocab.json',
 './gpt2-finetuned/merges.txt',
 './gpt2-finetuned/added_tokens.json')

In [10]:
# Load the fine-tuned model
from transformers import pipeline

generator = pipeline('text-generation', model="./gpt2-finetuned", tokenizer="./gpt2-finetuned")

# Test generation
prompt = "The future of technology is"
outputs = generator(prompt, max_length=100, num_return_sequences=1)

# Display the generated text
print(outputs[0]['generated_text'])


Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


The future of technology is also defined by changing perceptions, the ability of industries to invest in human capital needs, and the ability of policymakers to manage their investments in a timely fashion. The future of technology is also defined by changing perceptions, the ability of industries to invest in human capital needs, and the ability of policymakers to manage their investments in a timely fashion.

In 2010, a report by the Intergovernmental Panel on Climate Change (IPCC), the leading group of scientists and policy makers on
